# Training DDPM on CIFAR-10

This notebook trains the U-Net from [`04_unet.py`](04_unet.py) with Algorithm 1 from
[Ho et al. (2020)](https://arxiv.org/abs/2006.11239):

$$L_\text{simple} = \|\varepsilon - \varepsilon_\theta(x_t, t)\|_2^2.$$

The paper-width U-Net trains for 100,000 optimizer steps on CUDA. Training starts only after the shape, gradient, overfit, checkpoint, AMP, and memory checks below pass. Sampling and evaluation are in [`06_sample_eval.ipynb`](06_sample_eval.ipynb).

The model trains on the official 50,000-image training split and ignores labels. The test split is reserved for final evaluation.


## 1. Setup


In [ ]:
import copy
import importlib
import math
import os
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

sys.modules["diffusion"] = importlib.import_module("02_diffusion")
sys.modules["model"] = importlib.import_module("04_unet")
from diffusion import DiffusionSchedule, build_schedule, simple_loss
from model import UNet


@dataclass(frozen=True)
class Config:
    seed: int = 42
    data_dir: Path = Path("dataset")
    checkpoint_dir: Path = Path("checkpoints")
    image_size: int = 32
    in_channels: int = 3
    num_steps: int = 1000
    beta_start: float = 1e-4
    beta_end: float = 0.02
    base_channels: int = 128
    channel_mults: tuple[int, ...] = (1, 2, 2, 2)
    num_res_blocks: int = 2
    attention_resolutions: tuple[int, ...] = (16,)
    dropout: float = 0.1
    batch_size: int = 32
    grad_accum_steps: int = 4
    learning_rate: float = 2e-4
    ema_decay: float = 0.9999
    max_steps: int = 100_000
    log_every: int = 50
    checkpoint_every: int = 5_000


physical_batch = int(os.getenv("DDPM_BATCH_SIZE", "32"))
if physical_batch <= 0 or 128 % physical_batch:
    raise ValueError("DDPM_BATCH_SIZE must be a positive divisor of 128")
cfg = Config(batch_size=physical_batch, grad_accum_steps=128 // physical_batch)

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for training")

device = torch.device("cuda")
torch.cuda.manual_seed_all(cfg.seed)
scaler = torch.amp.GradScaler("cuda")
autocast_context = lambda: torch.autocast("cuda")
cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)

RUN_TRAINING = os.getenv("DDPM_RUN_TRAINING", "0") == "1"
RESUME_CHECKPOINT = cfg.checkpoint_dir / "ddpm_cifar10_production_latest.pt"
OVERFIT_CHECKPOINT = cfg.checkpoint_dir / "overfit_production.pt"
PROGRESS_LOG = cfg.checkpoint_dir / "training_progress.log"
hardware = torch.cuda.get_device_name(0)

print(f"PyTorch: {torch.__version__}; torchvision: {torchvision.__version__}; NumPy: {np.__version__}")
print(f"Device: {device} ({hardware})")
print(f"AMP: True; production training: {RUN_TRAINING}")
print(cfg)


## 2. Training data

Same transform as [`01_dataset.ipynb`](01_dataset.ipynb): random horizontal flip, then map
$[0, 1]$ to $[-1, 1]$. Labels are loaded and discarded.


In [ ]:
diffusion_transform = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

data_ready = (cfg.data_dir / "cifar-10-batches-py").exists()
train_dataset = datasets.CIFAR10(
    root=cfg.data_dir,
    train=True,
    download=not data_ready,
    transform=diffusion_transform,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=device.type == "cuda",
    drop_last=True,
)

images, labels = next(iter(train_loader))
print(f"Train images: {len(train_dataset):,}")
print(f"Batch shape: {tuple(images.shape)}")
print(f"Value range: [{images.min():.3f}, {images.max():.3f}]")
print(f"Ignoring {labels.numel()} class labels (unconditional DDPM).")

assert images.shape == (cfg.batch_size, cfg.in_channels, cfg.image_size, cfg.image_size)
assert images.min() >= -1.01 and images.max() <= 1.01


## 3. Schedule and U-Net

Imported from [`02_diffusion.py`](02_diffusion.py) and [`04_unet.py`](04_unet.py).


In [ ]:
schedule = build_schedule(cfg.num_steps, cfg.beta_start, cfg.beta_end).to(device)
model = UNet(cfg).to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parameters: {parameter_count:,} ({parameter_count / 1e6:.2f} M)")
print(f"alpha_bar: {schedule.alpha_bars[0].item():.6f} -> {schedule.alpha_bars[-1].item():.6e}")

assert 30e6 < parameter_count < 42e6
assert schedule.posterior_variance[0].item() == 0.0


## 4. $L_\text{simple}$

Sample $t$ uniformly, draw $\varepsilon \sim \mathcal{N}(0, I)$, form $x_t$ with `q_sample`, and
regress the U-Net onto that same $\varepsilon$. The loss lives in [`02_diffusion.py`](02_diffusion.py).
Zero-init on the output convolution makes the first loss close to $\mathrm{MSE}(0, \varepsilon) \approx 1$.


In [ ]:
probe_images = images[:2].to(device)
probe_timesteps = torch.tensor([0, cfg.num_steps - 1], device=device)
prediction = model(probe_images, probe_timesteps)
probe_loss = simple_loss(model, probe_images, schedule)
probe_loss.backward()
gradients = [parameter.grad for parameter in model.parameters() if parameter.grad is not None]
model.zero_grad(set_to_none=True)

print(f"U-Net: {tuple(probe_images.shape)} -> {tuple(prediction.shape)}")
print(f"First L_simple: {probe_loss.item():.4f}")
assert prediction.shape == probe_images.shape
assert torch.isfinite(prediction).all() and torch.isfinite(probe_loss)
assert gradients and all(torch.isfinite(gradient).all() for gradient in gradients)
assert 0.3 < probe_loss.item() < 3.0


## 5. One-batch overfit

This is the stop criterion from the report. We freeze one small batch and a fixed $(t, \varepsilon)$
so the graph has a single target. If this loss does not fall, the long run will not either.

A few extra steps with freshly sampled $t$ then confirm the stochastic training objective still runs.


In [ ]:
overfit_batch_size = 8
overfit_steps = 200

overfit_images = images[:overfit_batch_size].to(device)
overfit_timesteps = torch.linspace(0, cfg.num_steps - 1, overfit_batch_size, device=device).long()
overfit_noise = torch.randn_like(overfit_images)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
ema_model = copy.deepcopy(model)
ema_model.requires_grad_(False)
ema_model.eval()


@torch.no_grad()
def update_ema(ema_model: nn.Module, model: nn.Module, decay: float) -> None:
    for ema_param, param in zip(ema_model.parameters(), model.parameters()):
        ema_param.data.mul_(decay).add_(param.data, alpha=1.0 - decay)
    for ema_buffer, buffer in zip(ema_model.buffers(), model.buffers()):
        ema_buffer.copy_(buffer)


model.train()
overfit_losses = []
for step in range(1, overfit_steps + 1):
    optimizer.zero_grad(set_to_none=True)
    with autocast_context():
        loss = simple_loss(model, overfit_images, schedule, overfit_timesteps, overfit_noise)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    update_ema(ema_model, model, cfg.ema_decay)
    overfit_losses.append(float(loss.detach().cpu()))
    if step == 1 or step == overfit_steps or step % 10 == 0:
        print(f"overfit step {step:3d}/{overfit_steps}: L_simple={overfit_losses[-1]:.4f}")

start = sum(overfit_losses[:5]) / 5
end = sum(overfit_losses[-5:]) / 5
print(f"Fixed-target overfit: {start:.4f} -> {end:.4f}")
assert all(math.isfinite(value) for value in overfit_losses)
assert end < 0.5 * start, f"overfit did not drop enough: {start:.4f} -> {end:.4f}"

# Stochastic L_simple on the same images (random t, random noise), as in Algorithm 1.
model.train()
random_losses = []
for _ in range(8):
    optimizer.zero_grad(set_to_none=True)
    with autocast_context():
        loss = simple_loss(model, overfit_images, schedule)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    random_losses.append(float(loss.detach().cpu()))
print("Random-t losses on the same batch:", ", ".join(f"{value:.3f}" for value in random_losses))
assert all(math.isfinite(value) for value in random_losses)

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(range(1, len(overfit_losses) + 1), overfit_losses, color="#176b4d")
ax.set_xlabel("update")
ax.set_ylabel(r"$L_{\mathrm{simple}}$")
ax.set_title("One-batch overfit (fixed x0, t, noise)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 6. Checkpoints and resume

The checkpoint stores everything needed to continue: weights, EMA copy, optimizer, AMP scaler,
step counter, config, and loss history. After a save/load round-trip, a fresh model must match the
saved weights and take one more finite step.


In [ ]:
def config_payload(config: Config) -> dict:
    payload = asdict(config)
    payload["data_dir"] = str(config.data_dir)
    payload["checkpoint_dir"] = str(config.checkpoint_dir)
    return payload


def save_checkpoint(
    path: Path,
    model: nn.Module,
    ema_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    global_step: int,
    loss_history: list[float],
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model": model.state_dict(),
            "ema_model": ema_model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "global_step": global_step,
            "config": config_payload(cfg),
            "loss_history": loss_history,
        },
        path,
    )


def load_checkpoint(
    path: Path,
    model: nn.Module,
    ema_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
) -> tuple[int, list[float]]:
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    if checkpoint["config"] != config_payload(cfg):
        raise ValueError("checkpoint config does not match the production configuration")
    model.load_state_dict(checkpoint["model"])
    ema_model.load_state_dict(checkpoint["ema_model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scaler.load_state_dict(checkpoint["scaler"])
    return int(checkpoint["global_step"]), list(checkpoint["loss_history"])


save_checkpoint(
    OVERFIT_CHECKPOINT,
    model,
    ema_model,
    optimizer,
    scaler,
    global_step=overfit_steps,
    loss_history=overfit_losses,
)

# Fresh objects, then restore. If resume is wrong, parameter equality fails.
resumed_model = UNet(cfg).to(device)
resumed_ema = copy.deepcopy(resumed_model)
resumed_ema.requires_grad_(False)
resumed_optimizer = torch.optim.Adam(resumed_model.parameters(), lr=cfg.learning_rate)
resumed_scaler = torch.amp.GradScaler("cuda")
loaded_step, loaded_history = load_checkpoint(
    OVERFIT_CHECKPOINT, resumed_model, resumed_ema, resumed_optimizer, resumed_scaler
)

ref = next(model.parameters()).detach()
got = next(resumed_model.parameters()).detach()
ema_ref = next(ema_model.parameters()).detach()
ema_got = next(resumed_ema.parameters()).detach()
print(f"Loaded step {loaded_step}, {len(loaded_history)} loss values, file={OVERFIT_CHECKPOINT}")
assert loaded_step == overfit_steps
assert loaded_history == overfit_losses
assert torch.allclose(ref, got)
assert torch.allclose(ema_ref, ema_got)

resumed_model.train()
resumed_optimizer.zero_grad(set_to_none=True)
with autocast_context():
    resume_loss = simple_loss(resumed_model, overfit_images, schedule, overfit_timesteps, overfit_noise)
resumed_scaler.scale(resume_loss).backward()
resumed_scaler.step(resumed_optimizer)
resumed_scaler.update()
print(f"One step after resume: {resume_loss.item():.4f}")
assert torch.isfinite(resume_loss)

# Keep training on the restored objects for the (optional) long run.
model, ema_model, optimizer, scaler = resumed_model, resumed_ema, resumed_optimizer, resumed_scaler


## 7. Production training

Run once with training disabled to check CUDA, AMP, throughput, peak VRAM, EMA, and checkpoint restore. If batch 32 does not fit, set `DDPM_BATCH_SIZE=16`; accumulation preserves effective batch 128.

```bash
jupyter execute 05_train.ipynb --inplace
DDPM_RUN_TRAINING=1 jupyter execute 05_train.ipynb --inplace
```

The second command resumes `checkpoints/ddpm_cifar10_production_latest.pt` and stops at 100,000 optimizer steps.


In [ ]:
def infinite_loader(loader: DataLoader):
    while True:
        yield from loader


def train(model, ema_model, optimizer, scaler, schedule, loader, start_step, loss_history):
    model.train()
    batches = infinite_loader(loader)
    optimizer.zero_grad(set_to_none=True)
    running = 0.0
    micro_step = 0
    step = start_step
    started = time.perf_counter()

    while step < cfg.max_steps:
        batch_images, _ = next(batches)
        batch_images = batch_images.to(device, non_blocking=device.type == "cuda")
        with autocast_context():
            loss = simple_loss(model, batch_images, schedule) / cfg.grad_accum_steps
        scaler.scale(loss).backward()
        running += float(loss.detach().cpu())
        micro_step += 1
        if micro_step % cfg.grad_accum_steps:
            continue

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        update_ema(ema_model, model, cfg.ema_decay)
        step += 1
        loss_history.append(running)
        running = 0.0

        if step == start_step + 1 or step % cfg.log_every == 0:
            now = time.perf_counter()
            steps_per_second = (step - start_step) / (now - started)
            eta_minutes = (cfg.max_steps - step) / steps_per_second / 60
            message = (
                f"step {step}/{cfg.max_steps}  L_simple={loss_history[-1]:.4f}  "
                f"{steps_per_second:.2f} steps/s  ETA={eta_minutes:.0f} min"
            )
            print(message)
            with PROGRESS_LOG.open("a") as log_file:
                print(message, file=log_file)

        if step % cfg.checkpoint_every == 0 or step == cfg.max_steps:
            save_checkpoint(
                RESUME_CHECKPOINT,
                model,
                ema_model,
                optimizer,
                scaler,
                step,
                loss_history,
            )
            print(f"Wrote {RESUME_CHECKPOINT} at step {step}")

    return step, loss_history


def synchronize_device():
    torch.cuda.synchronize()


model.train()
bench_images = images.to(device, non_blocking=True)
n_bench = 3
torch.cuda.reset_peak_memory_stats()
synchronize_device()
bench_start = time.perf_counter()
for _ in range(n_bench):
    optimizer.zero_grad(set_to_none=True)
    for _ in range(cfg.grad_accum_steps):
        with autocast_context():
            bench_loss = simple_loss(model, bench_images, schedule) / cfg.grad_accum_steps
        scaler.scale(bench_loss).backward()
    scaler.step(optimizer)
    scaler.update()
synchronize_device()
steps_per_second = n_bench / (time.perf_counter() - bench_start)
print(f"Benchmark: {steps_per_second:.2f} optimizer steps/s on {hardware}")
print(f"Estimated 100k training: {cfg.max_steps / steps_per_second / 3600:.1f} hours")
print(f"Peak memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")

if RUN_TRAINING:
    start_step = 0
    history = []
    model = UNet(cfg).to(device)
    ema_model = copy.deepcopy(model).eval()
    ema_model.requires_grad_(False)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    scaler = torch.amp.GradScaler("cuda")

    if RESUME_CHECKPOINT.exists():
        start_step, history = load_checkpoint(RESUME_CHECKPOINT, model, ema_model, optimizer, scaler)
        print(f"Resuming at step {start_step}")
    else:
        print("Starting from random initialization")

    with PROGRESS_LOG.open("a") as log_file:
        print(f"start_step={start_step} hardware={hardware}", file=log_file)
    training_started = time.perf_counter()
    final_step, history = train(
        model,
        ema_model,
        optimizer,
        scaler,
        schedule,
        train_loader,
        start_step,
        history,
    )
    training_wall_minutes = (time.perf_counter() - training_started) / 60
    torch.save(
        {
            "ema_model": ema_model.state_dict(),
            "config": config_payload(cfg),
            "global_step": final_step,
            "loss_history": history,
        },
        cfg.checkpoint_dir / "ddpm_cifar10_production_ema.pt",
    )
    print(f"Training wall time: {training_wall_minutes:.1f} min")
    print(f"L_simple: {history[0]:.4f} -> {history[-1]:.4f} ({len(history)} updates)")
else:
    print("Production training skipped; set DDPM_RUN_TRAINING=1 to enable it.")


## 8. Checks completed

- The U-Net preserves image shape and has finite outputs, loss, and gradients.
- A fixed batch/noise target overfits, while random-timestep updates remain finite.
- Checkpoints restore model, EMA, optimizer, scaler, config, step, and loss history.
- CUDA preflight reports AMP throughput and peak VRAM before production training is enabled.

Next: [`06_sample_eval.ipynb`](06_sample_eval.ipynb).
